In [22]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from layers import MLP

### teacher model

In [ ]:
class JointEncoder(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim, # 각 feature의 출력층 dim
        enc_hidden_dim, # hidden layer의 dim
    ):
        super().__init__()
        self.num_features = num_features

        self.input_proj = nn.Linear(2, enc_hidden_dim) # token embedding [x * m, m]

        self.attention = nn.MultiheadAttention(
            embed_dim=enc_hidden_dim,
            num_heads=1,
            batch_first=True
        )

        self.out_proj = nn.Linear(enc_hidden_dim, z_dim)

    def forward(self, x, m):
        x_masked = x * m
        h = torch.stack([x_masked, m], dim=-1)
        token = self.input_proj(h) # token embedding

        attn_out, attn_weights = self.attention(
            token, # query
            token, # key
            token  # value
        )

        z_raw = self.out_proj(attn_out)
        return z_raw

In [ ]:
class TeacherModel(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim,
        enc_hidden_dim,
        dec_hidden_dim, # 디코더 hidden layer의 dim
        dec_num_hidden, # 디코더 hidden layer 개수
        out_dim # 클래수 개수 (regression에서는 1)
    ):
        super().__init__()
        dec_input_dim = num_features * z_dim

        self.encoder = JointEncoder(
            num_features=num_features,
            z_dim=z_dim,
            enc_hidden_dim=enc_hidden_dim,
        )

        self.predictor = MLP(
            in_dim=dec_input_dim,
            hidden_dim=dec_hidden_dim,
            out_dim=out_dim,
            num_hidden=dec_num_hidden
        )

    def forward(self, x, m):
        z = self.encoder(x, m)
        B, D, Z = z.shape # batch, feature dim, z dim
        z = z.view(B, D * Z)

        logit = self.predictor(z)
        return logit

In [25]:
# contrastive loss와의 확장을 위해 따로 미리 만들어둠
class TeacherLoss(nn.Module):
    def __init__(
            self, 
            teacher_model,
            task_type
        ):
        super().__init__()
        self.teacher = teacher_model
        self.task_type = task_type

        if task_type == "multi_classification":
            self.criterion = nn.CrossEntropyLoss()

        elif task_type == "binary_classification":
            self.criterion = nn.BCEWithLogitsLoss()

        elif task_type == "regression":
            self.criterion = nn.MSELoss()

    def forward(self, x, m, y):
        logit = self.teacher(x, m)

        if self.task_type == "multi_classification":
            loss = self.criterion(logit, y)

        elif self.task_type == "binary_classification":
            logit = logit.view(-1)
            y = y.view(-1).float()
            loss = self.criterion(logit, y)

        elif self.task_type == "regression":
            logit = logit.view(-1)
            y = y.view(-1).float()
            loss = self.criterion(logit, y)

        return {
            "loss": loss,
            "logit": logit,
        }

### student model

In [ ]:
class StudentModel(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim,
        enc_hidden_dim,
        dec_hidden_dim, # 디코더 hidden layer의 dim
        dec_num_hidden, # 디코더 hidden layer 개수
        out_dim # 클래수 개수 (regression에서는 1)
    ):
        super().__init__()
        dec_input_dim = num_features * z_dim

        self.encoder = JointEncoder(
            num_features=num_features,
            z_dim=z_dim,
            enc_hidden_dim=enc_hidden_dim,
        )

        self.predictor = MLP(
            in_dim=dec_input_dim,
            hidden_dim=dec_hidden_dim,
            out_dim=out_dim,
            num_hidden=dec_num_hidden
        )

    def forward(self, x, m):
        z_raw = self.encoder(x, m)
        z = z_raw * m.unsqueeze(-1)

        B, D, Z = z.shape
        z = z.view(B, D * Z)

        logit = self.predictor(z)
        return logit

In [27]:
class StudentLoss(nn.Module):
    def __init__(
      self,
      student_model,
      teacher_model,
      task_type,
      lambda_distill, # distill loss
      lambda_pred # prediction loss
    ):
        super().__init__()
        self.student = student_model
        self.teacher = teacher_model
        self.task_type = task_type
        self.lambda_distill = lambda_distill
        self.lambda_pred = lambda_pred
        self.mse = nn.MSELoss()

        if task_type == "multi_classification":
            self.criterion = nn.CrossEntropyLoss()

        elif task_type == "binary_classification":
            self.criterion = nn.BCEWithLogitsLoss()

        elif task_type == "regression":
            self.criterion = nn.MSELoss()

    def forward(self, x_full, x_masked, m_masked, y):
        '''
        x_full: teacher가 보는 full feature
        x_masked: student가 보는 masked feature
        m_masked: student mask (관측:1, 결측:0)
        '''
        # distill loss
        with torch.no_grad():
            m_full = torch.ones_like(x_full) # teacher는 mask가 모두 1
            z_teacher = self.teacher.encoder(x_full, m_full)
        
        z_student = self.student.encoder(x_masked, m_masked)
        logit_student = self.student(x_masked, m_masked)

        loss_distill = self.mse(z_student, z_teacher)

        # prediction loss
        if self.task_type == "multi_classification":
            loss_sup = self.criterion(logit_student, y)

        elif self.task_type == "binary_classification":
            logit_flat = logit_student.view(-1)
            y_flat = y.view(-1).float()
            loss_sup = self.criterion(logit_flat, y_flat)

        elif self.task_type == "regression":
            logit_flat = logit_student.view(-1)
            y_flat = y.view(-1).float()
            loss_sup = self.criterion(logit_flat, y_flat)

        # total loss
        loss = self.lambda_distill * loss_distill + self.lambda_pred * loss_sup

        return {
            "loss": loss,
            "loss_distill": loss_distill,
            "loss_sup": loss_sup,
            "logit_student": logit_student,
        }


In [ ]:
class JointFeatureAcquisition():
    def __init__(self, x, m, predictor, alpha=1, gamma=0):
        self.x = x
        self.m = m
        self.predictor = predictor
        self.alpha = alpha
        self.gamma = gamma

    def entropy(self, p,  eps=1e-10):
        alpha = self.alpha
        p = np.clip(p, eps, 1.0)  

        if alpha == 0.0:
            return np.log((p > eps).sum(axis=-1) + eps)
        
        elif alpha == 1.0:
            return -np.sum(p * np.log(p + eps), axis=-1)

        elif alpha > 1000:
            return -np.log(np.max(p, axis=-1) + eps)
        
        else:
            return (1.0 / (1.0 - alpha)) * np.log(np.sum(np.power(p, alpha), axis=-1) + eps)

    def alpha_gamma_cmi(self):
        x = self.x
        m = self.m
        gamma = self.gamma
        predictor = self.predictor

        device = next(predictor.parameters()).device

        m_upsampled = np.random.binomial(n=1, p=gamma, size=m.shape) # 각 feature별로 0 또는 1로 변형 
        m_repeated = np.maximum(m, m_upsampled) # 위에서는 모든 feature별로 진행했으니 max로 병합
        
        m_repeated = torch.tensor(m_repeated, dtype=torch.float32, device=device)

        with torch.no_grad():
            z_base = predictor.encoder(x, m_repeated) # 기본 z 얻어놓기
        B, D, Z = z_base.shape

        out = []

        # 변수 하나씩 mask해가며 entropy 계산
        for f in range(D):
            # without 계산
            m_without = m_repeated.clone()
            m_without[:, f] = 0.0
            z_without = z_base * m_without.unsqueeze(-1)

            with torch.no_grad():
                logits_without = predictor.predictor(z_without.view(B, D * Z))
                p_without = torch.softmax(logits_without, dim=-1).cpu().numpy()
            h_without = self.entropy(p=p_without)

            # with 계산
            m_with = m_repeated.clone()
            m_with[:, f] = 1.0
            z_with = z_base * m_with.unsqueeze(-1)

            with torch.no_grad():
                logits_with = predictor.predictor(z_with.view(B, D * Z))
                p_with = torch.softmax(logits_with, dim=-1).cpu().numpy()
            h_with = self.entropy(p=p_with)

            # 차이 계산
            entropy_diff = h_without - h_with
            out.append(entropy_diff)
            
        return np.stack(out, axis=-1)

    def acquire(self):
        m = self.m
        scores = self.alpha_gamma_cmi()
        scores -= scores.min()
        scores *= (1 - m)
        scores += 1e-10 * (1 - m) * np.random.uniform(size=(scores.shape)) # 최고점 score 점수 같음 방지

        selected = np.argmax(scores, axis=-1)
        m[np.arange(m.shape[0]), selected] = 1.0
        self.m = m # acquire 후 해당 feature의 mask = 1로 변경 

        return m, selected

### Cube

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [29]:
DATA_DIR = os.path.join("data", "cube")

X_train = torch.load(f"{DATA_DIR}/X_train_cdf.pt").float()
y_train = torch.load(f"{DATA_DIR}/y_train.pt").long()

X_val   = torch.load(f"{DATA_DIR}/X_val_cdf.pt").float()
y_val   = torch.load(f"{DATA_DIR}/y_val.pt").long()

X_test = torch.load(f"{DATA_DIR}/X_test_cdf.pt").float()
y_test = torch.load(f"{DATA_DIR}/y_test.pt").long()

In [ ]:
def sample_mask_uniform_K_per_sample(bs, d, min_K, max_K): # batch size, feature 개수, 최소 관측 샘플 수, 최대 관측 샘플 수
    m = np.zeros((bs, d), dtype=np.float32)
    Ks = np.random.randint(min_K, max_K+1, size=(bs,))
    for i, K in enumerate(Ks): # Ks의 index와 해당 index의 값
        idx = np.random.choice(d, size=K, replace=False)
        m[i, idx] = 1.0
    return m

In [ ]:
num_features = 20
num_classes = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 256

train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val,   y_val)
test_ds  = TensorDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)


In [ ]:
z_dim          = 16   # per-feature latent dim
enc_hidden_dim = 64   # JointEncoder 내부 attention token dim
dec_hidden_dim = 128  # predictor MLP hidden dim
dec_num_hidden = 2    # predictor MLP hidden layer 개수

teacher = TeacherModel(
    num_features=num_features,
    z_dim=z_dim,
    enc_hidden_dim=enc_hidden_dim,
    dec_hidden_dim=dec_hidden_dim,
    dec_num_hidden=dec_num_hidden,
    out_dim=num_classes,
).to(device)

student = StudentModel(
    num_features=num_features,
    z_dim=z_dim,
    enc_hidden_dim=enc_hidden_dim,
    dec_hidden_dim=dec_hidden_dim,
    dec_num_hidden=dec_num_hidden,
    out_dim=num_classes,
).to(device)

teacher_loss_fn = TeacherLoss(
    teacher_model=teacher,
    task_type="multi_classification",
)

student_loss_fn = StudentLoss(
    student_model=student,
    teacher_model=teacher,
    task_type="multi_classification",
    lambda_distill=1.0,
    lambda_pred=1.0,
)

In [ ]:
def evaluate_classifier(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)

            # Teacher는 full mask (모든 feature 관측) 가정
            m_full = torch.ones_like(x, device=device)
            logits = model(x, m_full)
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    acc = correct / total if total > 0 else 0.0
    return acc

In [ ]:
num_epochs_teacher = 20
lr_teacher = 1e-3

optimizer_teacher = torch.optim.Adam(teacher.parameters(), lr=lr_teacher)

In [ ]:
for epoch in range(1, num_epochs_teacher + 1):
    teacher.train()
    total_loss = 0.0
    total_batches = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        # teacher는 full mask로 학습 (모든 feature 사용)
        m_full = torch.ones_like(x, device=device)

        out = teacher_loss_fn(x, m_full, y)
        loss = out["loss"]

        optimizer_teacher.zero_grad()
        loss.backward()
        optimizer_teacher.step()

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    val_acc = evaluate_classifier(teacher, val_loader, device)
    print(f"[Teacher] Epoch {epoch:02d} | loss={avg_loss:.4f} | val_acc={val_acc:.4f}")

# teacher는 학습 끝났으니 grad 갱신 안 하도록 freeze (선택)
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

In [ ]:
num_epochs_student = 20
lr_student = 1e-3

optimizer_student = torch.optim.Adam(student.parameters(), lr=lr_student)

In [ ]:
# student가 보는 feature 개수 범위 설정
min_K = 1   # 최소 관측 feature 수
max_K = 20  # 최대 관측 feature 수 (= num_features)

print("\n=== Train Student (random partial masks) ===")
for epoch in range(1, num_epochs_student + 1):
    student.train()
    total_loss = 0.0
    total_distill = 0.0
    total_sup = 0.0
    total_batches = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        B, D = x.shape

        # student용 random mask 샘플링 (numpy -> torch)
        m_np = sample_mask_uniform_K_per_sample(
            bs=B, d=D,
            min_K=min_K, max_K=max_K
        )  # (B, D) numpy
        m_masked = torch.tensor(m_np, dtype=torch.float32, device=device)  # (B, D)

        x_masked = x * m_masked  # student가 실제로 보는 입력 값

        out = student_loss_fn(
            x_full=x,           # teacher가 보는 full feature
            x_masked=x_masked,  # student가 보는 masked feature
            m_masked=m_masked,  # student mask
            y=y,
        )
        loss = out["loss"]
        loss_distill = out["loss_distill"]
        loss_sup = out["loss_sup"]

        optimizer_student.zero_grad()
        loss.backward()
        optimizer_student.step()

        total_loss += loss.item()
        total_distill += loss_distill.item()
        total_sup += loss_sup.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_distill = total_distill / max(total_batches, 1)
    avg_sup = total_sup / max(total_batches, 1)

    # student 평가: 여기서는 teacher와 동일하게 full mask 기준으로 accuracy 측정할 수도 있고,
    # 실제 시나리오처럼 일부 mask로 평가할 수도 있음.
    # 일단 비교를 위해 full mask로 평가해보자.
    student.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val_batch, y_val_batch in val_loader:
            x_val_batch = x_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            m_full_val = torch.ones_like(x_val_batch, device=device)
            logits_val = student(x_val_batch, m_full_val)
            preds_val = logits_val.argmax(dim=-1)
            correct += (preds_val == y_val_batch).sum().item()
            total += y_val_batch.size(0)
    val_acc_student = correct / total if total > 0 else 0.0

    print(f"[Student] Epoch {epoch:02d} | loss={avg_loss:.4f} "
          f"(distill={avg_distill:.4f}, sup={avg_sup:.4f}) | val_acc(full mask)={val_acc_student:.4f}")

In [ ]:
teacher_test_acc = evaluate_classifier(teacher, test_loader, device)

student.eval()
correct = 0
total = 0
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        m_full_test = torch.ones_like(x_batch, device=device)
        logits = student(x_batch, m_full_test)
        preds = logits.argmax(dim=-1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
student_test_acc = correct / total if total > 0 else 0.0

print("\n=== Final Test Accuracy ===")
print(f"Teacher (full mask)  test_acc = {teacher_test_acc:.4f}")
print(f"Student (full mask)  test_acc = {student_test_acc:.4f}")